# Funktion definieren um Konzentration für jeden Zeitschritt zu berechnen

In [2]:
subject = 'P08' # 'P08' 'Synthetic"  einziger Parameter der verändert werden sollte

#COMPARISONS = ['No_Lesion_Deep', 'No_Lesion_LR8', 'No_Lesion_GT']

#COMPARISONS = ['Lesion_Double_deep_tMPPCA_5D', 'Lesion_Double_tMPPCA_5D', 'Lesion_Double_LR8', 'Lesion_Double_GT'] #Lesion_Double_LR8

#COMPARISONS = ['Lesion_Double_GT', 'Lesion_Double_deep_tMPPCA_5D', 'Lesion_Double_tMPPCA_5D']

#COMPARISONS = ['Tumor_1_deep', 'Tumor_1_noisy', 'Tumor_1_Part_1_deep_tMPPCA_5D_Ynet']

COMPARISONS = ['P08_noisy', 'P08_tMPPCA_5D', "P08_deep_tMPPCA_5D"] #'P08_tMPPCA_5D',  'P08_fit_LR8'

quality_clip = False # show voxels that meet LC Model quality criteria
outlier_clip = False

# Available methods:
# - P08_noDenoising
# - P08_LR8
# - P08_unet_JInvariant

# - P08_simulated_GT   -- that is ground truth
# - P08_simulated_noisy
# - P08_simulated_LR8
# - P08_simulated_unet_JInvariant

In [ ]:
"""timecourse_plot.py
-----------------------------------------------------------------
Plot-Funktion für PVE‑Zeitverläufe beliebiger Metabolite & Methoden.
Baselines werden **nicht** automatisch hinzugefügt – einfach gewünschte
Methodenliste übergeben (z. B. auch 'NoDenoising' als normale Methode).
"""

from __future__ import annotations
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls          # nicht‑negatives LS
from numpy.linalg import lstsq, inv

# -----------------------------------------------------------------------------
# -----------------   Hilfsfunktionen   ---------------------------------------
# -----------------------------------------------------------------------------

def fit_pve_single_metabolite(
    metabo_map: np.ndarray,           # (X,Y,Z,T)
    tissue_maps: np.ndarray,          # (X,Y,Z,3)  GM, WM, CSF
    brain_mask: np.ndarray | None = None,   # (X,Y,Z) oder (X,Y,Z,T)
    *,
    method: str = "nnls"              # "nnls" (default) oder "lstsq"
) -> tuple[np.ndarray, np.ndarray]:
    """Schätzt Metaboliten-Konzentrationen in reinem WM, GM, CSF über die Zeit.
    Rückgabe: conc[3,T], conc_std[3,T]"""

    gm, wm, csf = (tissue_maps[..., i].ravel() for i in range(3))

    # metabo_map muss 4D sein
    if metabo_map.ndim != 4:
        raise ValueError(f"metabo_map must be (X,Y,Z,T), got {metabo_map.shape}")
    T = metabo_map.shape[-1]

    # brain_mask: None -> alles True; 3D -> broadcast; 4D -> ok
    if brain_mask is None:
        brain_mask = np.ones(metabo_map.shape[:-1], dtype=bool)  # (X,Y,Z)
        brain_mask = brain_mask[..., None] * np.ones((1,1,1,T), dtype=bool)
    else:
        brain_mask = np.asarray(brain_mask, dtype=bool)
        if brain_mask.ndim == 3:
            brain_mask = brain_mask[..., None] * np.ones((1,1,1,T), dtype=bool)
        elif brain_mask.ndim != 4:
            raise ValueError(f"brain_mask must be 3D or 4D, got {brain_mask.shape}")

    # Flatten tissue A einmal (ohne Maske); Maske kommt pro t
    A_all = np.vstack([wm, gm, csf]).T  # (nVox, 3)
    Y_all = metabo_map.reshape(-1, T) / 1e7

    conc, conc_std = np.empty((3, T)), np.empty((3, T))

    # Für lstsq: wir brauchen AtA_inv pro t (weil Masken variieren!)
    # -> daher kein globales AtA_inv mehr
    for t in range(T):
        bm = brain_mask[..., t].ravel()
        A_full = A_all[bm]
        y_full = Y_all[bm, t]

        valid = np.isfinite(y_full) & (y_full != 0)
        A = A_full[valid]
        y = y_full[valid]

        if len(y) < 3:
            conc[:, t] = np.nan
            conc_std[:, t] = np.nan
            continue

        if method == "nnls":
            x, _ = nnls(A, y)
            res  = y - A @ x
            cov  = np.linalg.pinv(A.T @ A)
        else:
            x, *_ = lstsq(A, y, rcond=None)
            res  = y - A @ x
            cov  = np.linalg.pinv(A.T @ A)

        dof     = max(len(y) - 3, 1)
        sigma2  = (res @ res) / dof
        conc[:, t]     = x
        conc_std[:, t] = np.sqrt(sigma2 * np.diag(cov))

    return conc, conc_std



def _suffix(quality_clip: bool, outlier_clip: bool) -> str:
    """Ermittelt Dateisuffix gemäß Clip‑Flags."""
    return "OutlierClip" if outlier_clip else ("QualityClip" if quality_clip else "Orig")


def _load_amp(metabolite: str, method: str, suffix: str, data_dir: str) -> np.ndarray:
    """Lädt AMP‑Map. Erwartet MetabMaps/<method>/<metabolite>_amp_<method>_<suffix>.npy"""
    path = os.path.join(data_dir, method, f"{metabolite}_amp_{method}_{suffix}.npy")
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Datei fehlt: {path}")
    return np.load(path)

# -----------------------------------------------------------------------------
# -----------------   Hauptfunktion   -----------------------------------------
# -----------------------------------------------------------------------------

def plot_timecourse_metabolite(
    metabolite: str,
    methods: list[str],
    *,
    subject: str = "P08",
    quality_clip: bool = False,
    outlier_clip: bool = False,
    data_dir: str = "MetabMaps",
    tissue_dir: str = "../datasets",
    colors: tuple[str, ...] | None = None,
    normalize_to_water: bool = True,   # <-- NEU
    water_name: str = "water",         # <-- NEU (falls deine Datei anders heißt)
    eps: float = 1e-8,                 # <-- NEU
    crlb_threshold: float | None = None,   # <-- NEU
    crlb_source_method: str | None = None, # <-- NEU (default: methods[0])

) -> None:

    # ---------------- Tissue-Segmentierung laden ----------------------------
    seg_path  = os.path.join(tissue_dir, subject, "gm_wm_csf_segmentation.npy")
    mask_path = os.path.join(tissue_dir, subject, "mask.npy")
    segmentations = np.swapaxes(np.load(seg_path), 0, -2)           # (X,Y,Z,3)
    brain_mask    = np.swapaxes(np.load(mask_path), 0, -1) > 0      # (X,Y,Z)

        # ---------------- CRLB Mask (optional) ---------------------------------
    # Wenn crlb_threshold gesetzt ist, machen wir eine 4D Maske:
    # (brain_mask[...,None] & (CRLB <= thr) & finite etc.)
    if crlb_source_method is None:
        crlb_source_method = methods[0]  # typischerweise noisy

    crlb_mask_4d = None
    if crlb_threshold is not None:
        # Lade "sd" Map (CRLB Proxy) für genau diesen Metaboliten aus der Quelle
        # erwartet: MetabMaps/<method>/<metabolite>_sd_<method>_<suffix>.npy
        crlb_path = os.path.join(data_dir, crlb_source_method,
                                 f"{metabolite}_sd_{crlb_source_method}_{suffix}.npy")
        if not os.path.isfile(crlb_path):
            raise FileNotFoundError(f"CRLB/SD Datei fehlt: {crlb_path}")

        crlb = np.load(crlb_path).astype(np.float32, copy=False)   # (X,Y,Z,T)
        # valid mask wie in deinem Notebook:
        crlb_valid = np.isfinite(crlb) & (crlb < 900)
        crlb_ok    = crlb_valid & (crlb <= crlb_threshold)

        # brain_mask ist 3D -> auf 4D broadcasten
        bm4 = brain_mask[..., None] * np.ones((1,1,1,crlb.shape[-1]), dtype=bool)

        crlb_mask_4d = bm4 & crlb_ok

    # ---------------- AMP-Maps laden ---------------------------------------
    suffix = _suffix(quality_clip, outlier_clip)
    amp_maps, ok_methods = [], []

    for m in methods:
        try:
            amp = _load_amp(metabolite, m, suffix, data_dir)

            if normalize_to_water:
                w = _load_amp(water_name, m, suffix, data_dir)

                # Sicherheitschecks
                if amp.shape != w.shape:
                    raise ValueError(f"Shape mismatch: {metabolite} {amp.shape} vs {water_name} {w.shape} for method {m}")

                # Voxelweise Division, robust gegen 0/NaN
                denom = w.astype(np.float64)
                num   = amp.astype(np.float64)

                valid = np.isfinite(num) & np.isfinite(denom) & (np.abs(denom) > eps)
                out = np.full_like(num, np.nan, dtype=np.float64)
                out[valid] = num[valid] / denom[valid]

                amp = out.astype(np.float32)

            amp_maps.append(amp)
            ok_methods.append(m)

        except (FileNotFoundError, ValueError) as e:
            print(e)

    if not amp_maps:
        raise RuntimeError("Keine AMP-Dateien gefunden – prüfe Methoden & Suffix!")

    # ---------------- PVE-Fitting ------------------------------------------
    conc_all, conc_sd_all = [], []
    for amp in amp_maps:
        c, c_sd = fit_pve_single_metabolite(
            amp,
            segmentations,
            brain_mask=(crlb_mask_4d if crlb_mask_4d is not None else brain_mask),
            method="nnls",
        )
        conc_all.append(c)
        conc_sd_all.append(c_sd)

        # ---------------- Plot --------------------------------------------------
    tab_colors = plt.cm.get_cmap("tab10").colors
    if colors is None:
        colors = tuple(tab_colors[i % 10] for i in range(len(ok_methods)))

    T = conc_all[0].shape[1]
    time = np.arange(T)

    # Referenz = erste Methode
    ref_name = ok_methods[0]
    ref_c    = conc_all[0]
    ref_sd   = conc_sd_all[0]

    # Alle anderen Methoden bekommen je ein Fenster
    plot_methods = ok_methods[1:]
    plot_conc    = conc_all[1:]
    plot_sd      = conc_sd_all[1:]

    n_methods = len(plot_methods)
    if n_methods == 0:
        raise RuntimeError("Es gibt nur eine Methode – keine Vergleichsplots möglich.")

    fig, axes = plt.subplots(
        2, n_methods,
        figsize=(4.2 * n_methods, 8.4),
        sharex=True
    )

    if n_methods == 1:
        axes = np.array(axes).reshape(2, 1)

    for j, (m_name, m_c, m_sd) in enumerate(zip(plot_methods, plot_conc, plot_sd)):
        ax_tw = axes[0, j]
        ax_q  = axes[1, j]

        ax_tw.set_box_aspect(1)
        ax_q.set_box_aspect(1)

        # ---------- Referenz plotten ----------
        ref_col = "k"
        ref_gm, ref_wm = ref_c[0], ref_c[1]
        ref_gm_sd, ref_wm_sd = ref_sd[0], ref_sd[1]

        ax_tw.errorbar(time, ref_wm, yerr=ref_wm_sd,
                       fmt="o", color=ref_col, capsize=3, markersize=4,
                       linestyle="-", linewidth=1.2,
                       label=f"{ref_name} (WM)")

        ax_tw.errorbar(time, ref_gm, yerr=ref_gm_sd,
                       fmt="o", color=ref_col, capsize=3, markersize=4,
                       linestyle="--", linewidth=1.2,
                       label=f"{ref_name} (GM)")

        ref_ratio = ref_gm / (ref_wm + eps)
        ref_ratio_sd = ref_ratio * np.sqrt(
            (ref_gm_sd / (ref_gm + eps))**2 +
            (ref_wm_sd / (ref_wm + eps))**2
        )

        ax_q.errorbar(time, ref_ratio, yerr=ref_ratio_sd,
                      fmt="o", color=ref_col, capsize=3, markersize=4,
                      linestyle="none",
                      label=ref_name)

        # ---------- Vergleichsmethode ----------
        col = colors[j + 1]
        gm, wm = m_c[0], m_c[1]
        gm_sd, wm_sd = m_sd[0], m_sd[1]

        ax_tw.errorbar(time, wm, yerr=wm_sd,
                       fmt="o", color=col, capsize=3, markersize=4,
                       linestyle="-", linewidth=1.5,
                       label=f"{m_name} (WM)")

        ax_tw.errorbar(time, gm, yerr=gm_sd,
                       fmt="o", color=col, capsize=3, markersize=4,
                       linestyle="--", linewidth=1.5,
                       label=f"{m_name} (GM)")

        ratio = gm / (wm + eps)
        ratio_sd = ratio * np.sqrt(
            (gm_sd / (gm + eps))**2 +
            (wm_sd / (wm + eps))**2
        )

        ax_q.errorbar(time, ratio, yerr=ratio_sd,
                      fmt="o", color=col, capsize=3, markersize=4,
                      linestyle="none",
                      label=m_name)

        # ---------- Achsen / Layout ----------
        ax_tw.set_title(f"{m_name} vs {ref_name}")
        ax_tw.grid(True)
        ax_q.grid(True)

        if j == 0:
            ax_tw.set_ylabel("Konzentration (a.u.)")
            ax_q.set_ylabel("GM / WM")
        else:
            ax_tw.set_yticklabels([])
            ax_q.set_yticklabels([])

        ax_q.set_xlabel("Zeitpunkt")
        ax_tw.legend(fontsize=8, loc="upper left")

    fig.suptitle(
        f"{metabolite}-Zeitverlauf ({suffix})"
        + (" / normalized to water" if normalize_to_water else ""),
        fontsize=14
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [4]:
plot_timecourse_metabolite(
        metabolite="water",
        methods=COMPARISONS,
        quality_clip=quality_clip,
        outlier_clip=outlier_clip,
        subject = subject,
        crlb_threshold=30.0,
    )

plot_timecourse_metabolite(
        metabolite="Glc",
        methods=COMPARISONS,
        quality_clip=quality_clip,
        outlier_clip=outlier_clip,
        subject = subject,
    )

plot_timecourse_metabolite(
        metabolite="Glx",
        methods=COMPARISONS,
        quality_clip=quality_clip,
        outlier_clip=outlier_clip,
        subject = subject,
    )

plot_timecourse_metabolite(
        metabolite="Lac",
        methods=COMPARISONS,
        quality_clip=quality_clip,
        outlier_clip=outlier_clip,
        subject = subject,
    )

UnboundLocalError: cannot access local variable 'suffix' where it is not associated with a value